[← 02 - Aggregation](<02 - Aggregation (GROUP BY, HAVING, COUNT-SUM-AVG).ipynb>) · [Course Overview](<00 - Course Overview.ipynb>)

# 03 - JOINs (INNER vs LEFT)

`02 - Aggregation` summarized rows from a single table. Real questions usually need two tables at once, a customer and their orders, an employee and their department. That's what a JOIN is for.

*New here? See the [Course Overview](<00 - Course Overview.ipynb>) for how to actually run these notebooks, viewing this on GitHub shows a preview, it won't execute the SQL.*

> **By the end of this notebook you'll be able to:** combine rows from two tables with `INNER JOIN` and `LEFT JOIN`, and explain why joining tables can multiply your row count in a way that quietly breaks a `COUNT` or `SUM` if you're not paying attention.

## 1. INNER JOIN: only rows that match on both sides

`Sales.Customer` has one row per customer. `Sales.SalesOrderHeader` has one row per order, with a `CustomerID` column pointing back to the customer who placed it. `INNER JOIN` returns a row only when a match exists on both sides.

**Example:**

In [ ]:
SELECT c.CustomerID, c.AccountNumber, s.SalesOrderID, s.OrderDate
FROM Sales.Customer c
INNER JOIN Sales.SalesOrderHeader s
    ON c.CustomerID = s.CustomerID;

Notice what's missing: any customer who has never placed an order doesn't appear at all. `INNER JOIN` silently drops non-matching rows from both sides, which is exactly right when you only want customers with orders, and exactly wrong when you're trying to count all customers.

## 2. LEFT JOIN: keep every row from the left table

`LEFT JOIN` keeps every row from the left table (`Sales.Customer`), whether or not it finds a match on the right. Where there's no match, the right side's columns come back as `NULL`.

**Example:**

In [ ]:
SELECT c.CustomerID, c.AccountNumber, s.SalesOrderID, s.OrderDate
FROM Sales.Customer c
LEFT JOIN Sales.SalesOrderHeader s
    ON c.CustomerID = s.CustomerID
WHERE s.SalesOrderID IS NULL;
-- Customers with no matching order at all: s.SalesOrderID is NULL for every column that came from the right table

## 3. Why grain matters

Here's the part that catches people after they already know the syntax. `Sales.Customer` has one row per customer. Once you join it to `Sales.SalesOrderHeader`, the result has one row per **order**, not per customer. A customer with five orders shows up five times.

<p align="center">
  <img src="graphics/03_join_grain.png" width="500" alt="Joining multiplies rows to match the other table's grain">
</p>

This is exactly why `SELECT COUNT(*) FROM Sales.Customer c INNER JOIN Sales.SalesOrderHeader s ON c.CustomerID = s.CustomerID` does **not** tell you how many customers have orders, it tells you how many orders exist across those customers. If you actually want a per-customer count, you need `GROUP BY` back on top of the join:

```sql
SELECT c.CustomerID, COUNT(s.SalesOrderID) AS "Order Count"
FROM Sales.Customer c
INNER JOIN Sales.SalesOrderHeader s
    ON c.CustomerID = s.CustomerID
GROUP BY c.CustomerID;
```

Joining and then aggregating without checking which table's grain you ended up at is one of the most common ways a report quietly overcounts something.

## What's Next

You can now combine two tables and reason about what a join does to your row count. Next: `04 - Subqueries to CTEs`, where naming intermediate steps makes a multi-table query like the ones above much easier to read.

---

[← 02 - Aggregation](<02 - Aggregation (GROUP BY, HAVING, COUNT-SUM-AVG).ipynb>) · [Course Overview](<00 - Course Overview.ipynb>) · [04 - CTEs →](<04 - Subqueries to CTEs.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).